In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### **Data Load**

In [2]:
df = pd.read_csv(r"C:\Users\QUIKCARE COMPUTERS\Downloads\archive (9)\rapidoBookings.csv")

### **Create Working Copy**

In [3]:
df_clean = df.copy()

### **Basic Data Inspection**

In [4]:
print(df_clean.shape)
print(df_clean.head())
print(df_clean.info())

(150000, 21)
         Date      Time    Booking ID   Booking Status   Customer ID  \
0  2024-03-23  12:29:38  "CNR5884300"  No Driver Found  "CID1982111"   
1  2024-11-29  18:01:39  "CNR1326809"       Incomplete  "CID4604802"   
2  2024-08-23  08:56:10  "CNR8494506"        Completed  "CID9202816"   
3  2024-10-21  17:17:25  "CNR8906825"        Completed  "CID2610914"   
4  2024-09-16  22:08:00  "CNR1950162"        Completed  "CID9933542"   

    Vehicle Type      Pickup Location      Drop Location  Avg VTAT  Avg CTAT  \
0          eBike          Palam Vihar            Jhilmil       NaN       NaN   
1       Go Sedan        Shastri Nagar  Gurgaon Sector 56       4.9      14.0   
2           Auto              Khandsa      Malviya Nagar      13.4      25.8   
3  Premier Sedan  Central Secretariat           Inderlok      13.1      28.5   
4           Bike     Ghitorni Village        Khan Market       5.3      19.6   

   ...  Reason for cancelling by Customer Cancelled Rides by Driver  \
0 

### **Data Cleaning**
### Date and Time Conversion

In [5]:
df_clean["DateTime"] = pd.to_datetime(
    df_clean["Date"].astype(str) + " " +
    df_clean["Time"].astype(str),
    errors="coerce")

### **Validation**

In [6]:
print("Invalid DateTime:",
    df_clean["DateTime"].isna().sum())

Invalid DateTime: 0


### **Create Date and Time Features**
### These Features allow us to analyze:
- Monthly Demand
- Daily Demand
- Weekday Demand
- Peak Hours

In [7]:
df_clean["Year"] = df_clean["DateTime"].dt.year
df_clean["Month"] = df_clean["DateTime"].dt.month
df_clean["Month_Name"] = df_clean["DateTime"].dt.month_name()
df_clean["Day"] = df_clean["DateTime"].dt.day
df_clean["Day_Name"] = df_clean["DateTime"].dt.day_name()
df_clean["Hour"] = df_clean["DateTime"].dt.hour

### **Duplicate Analysis**
### Extract Duplicate Rows

In [8]:
print("Exact duplicate rows:",
    df_clean.duplicated().sum())

Exact duplicate rows: 0


### Duplicate Booking IDs

In [9]:
duplicate_booking_ids = df_clean[
    df_clean["Booking ID"].duplicated(keep=False)
].sort_values("Booking ID")

print(duplicate_booking_ids.head(20))

              Date      Time    Booking ID         Booking Status  \
81334   2024-10-15  18:17:23  "CNR1026036"              Completed   
9192    2024-07-21  17:59:41  "CNR1026036"        No Driver Found   
9587    2024-12-17  19:19:02  "CNR1029172"              Completed   
1353    2024-01-19  17:00:57  "CNR1029172"             Incomplete   
82029   2024-11-05  16:50:20  "CNR1051228"              Completed   
110412  2024-01-31  08:42:04  "CNR1051228"              Completed   
87333   2024-04-20  23:16:21  "CNR1056023"              Completed   
120008  2024-07-07  17:18:49  "CNR1056023"             Incomplete   
71570   2024-02-12  19:02:47  "CNR1058956"  Cancelled by Customer   
38548   2024-10-15  13:33:55  "CNR1058956"              Completed   
92822   2024-03-24  11:11:19  "CNR1071255"    Cancelled by Driver   
94087   2024-03-21  07:35:07  "CNR1071255"              Completed   
23113   2024-02-12  12:58:19  "CNR1076471"              Completed   
18828   2024-10-07  20:06:47  "CNR

In [10]:
print("Repeated Booking IDs:",
    df_clean["Booking ID"].duplicated().sum())

Repeated Booking IDs: 1233


### We do not delete these duplicate records.
Because investigation showed that repeated Booking IDs can have different dates, customers, vehicles and booking statuses.

Booking Id is not unique

In [11]:
df_clean["Ride_ID"] = range(1, len(df_clean) + 1)

### We create Ride Id because this is unique identifier

### **Missing Valuse Analysis**

In [12]:
missing_values = pd.DataFrame({
    "Missing_Count": df_clean.isnull().sum(),
    "Missing_Percentage":
        (df_clean.isnull().mean() * 100).round(2)})

print(missing_values)

                                   Missing_Count  Missing_Percentage
Date                                           0                 0.0
Time                                           0                 0.0
Booking ID                                     0                 0.0
Booking Status                                 0                 0.0
Customer ID                                    0                 0.0
Vehicle Type                                   0                 0.0
Pickup Location                                0                 0.0
Drop Location                                  0                 0.0
Avg VTAT                                   10500                 7.0
Avg CTAT                                   48000                32.0
Cancelled Rides by Customer               139500                93.0
Reason for cancelling by Customer         139500                93.0
Cancelled Rides by Driver                 123000                82.0
Driver Cancellation Reason        

### Important finding
### Many missing values are structural, not necessarily errors.
### So we do not fill with 0 and do not drop this because we will remove a large amount of valid business information.

### **Categorical Data Cleaning**

In [13]:
categorical_columns = [
    "Booking Status", "Vehicle Type",
    "Pickup Location", "Drop Location",
    "Payment Method"]

for col in categorical_columns:
    df_clean[col] = (df_clean[col].astype("string").str.strip())

In [14]:
print(df_clean["Booking Status"].unique())
print(df_clean["Vehicle Type"].unique())
print(df_clean["Payment Method"].unique())

<StringArray>
[      'No Driver Found',            'Incomplete',             'Completed',
   'Cancelled by Driver', 'Cancelled by Customer']
Length: 5, dtype: string
<StringArray>
['eBike', 'Go Sedan', 'Auto', 'Premier Sedan', 'Bike', 'Go Mini', 'Uber XL']
Length: 7, dtype: string
<StringArray>
[<NA>, 'UPI', 'Debit Card', 'Cash', 'Uber Wallet', 'Credit Card']
Length: 6, dtype: string


### **Numeric Validation**

In [15]:
numeric_columns = [
    "Avg VTAT",  "Avg CTAT",
    "Booking Value",  "Ride Distance",
    "Driver Ratings",  "Customer Rating"]

for col in numeric_columns:
    print(col, "Negative:", (df_clean[col] < 0).sum())

Avg VTAT Negative: 0
Avg CTAT Negative: 0
Booking Value Negative: 0
Ride Distance Negative: 0
Driver Ratings Negative: 0
Customer Rating Negative: 0


### **Rating Validation**

In [16]:
print("Invalid Driver Ratings:",
    ((df_clean["Driver Ratings"] < 1) |
        (df_clean["Driver Ratings"] > 5)).sum())

print("Invalid Customer Ratings:",
    ((df_clean["Customer Rating"] < 1) |
        (df_clean["Customer Rating"] > 5)).sum())

Invalid Driver Ratings: 0
Invalid Customer Ratings: 0


### Payment Method Clean

In [17]:
df_clean["Payment Method Clean"] = (
    df_clean["Payment Method"].fillna("Not Applicable"))

In [18]:
df_clean.to_csv("rapido_rides.csv", index=False)